In [ ]:
import torch
import torch.autograd as autograd         # computation graph
from torch import Tensor                  # tensor node in the computation graph
import torch.nn as nn                     # neural networks
import torch.optim as optim               # optimizers e.g. gradient descent, ADAM, etc.

import numpy as np
import math
import time
#from pyDOE import lhs         #Latin Hypercube Sampling
import scipy.io
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as cls

# Device configuration
device = torch.device('cpu')

In [ ]:
torch.set_default_dtype(torch.float64)

class SNN(nn.Module):

    def __init__(self, layers1, device):
        super().__init__() #call __init__ from parent class
        self.layers1 = layers1
        self.device = device
        self.activation = nn.Tanh()
        self.loss_function = nn.MSELoss(reduction ='mean')
        self.linears1 = nn.ModuleList([
            nn.Linear(layers1[i], layers1[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers1) - 1)
        ])
        self.M = layers1[-1]
        self.omega = torch.ones((self.M, 1), dtype=torch.float64).to(device)
        self.alpha = 1e-2
        self.a = 0
        self.b = 1
        

        'Xavier Normal Initialization'
        for i1 in range(len(layers1)-1):
            nn.init.xavier_normal_(self.linears1[i1].weight.data, gain=1.0)
            nn.init.zeros_(self.linears1[i1].bias.data)
            
    
        

    'foward pass'
    def forward(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x - self.a) * (self.b - x), 1)[:,None]
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers1)-2):
            z = self.linears1[i](x)
            x = self.activation(z)
        
        x = self.linears1[-1](x) #+ x_sum

        output = l_x *  x


        return output
                                
    def exact_solution(self,xy):
        x, y = xy[:, 0][:,None], xy[:, 1][:,None]
        temp = torch.sin(math.pi* x)* torch.sin(math.pi* y)
        u_exact = 1 / (1 + 4 * self.alpha * math.pow(math.pi,4)) * temp
        y_exact = 1 / (2 * math.pow(math.pi,2)) * u_exact
        p_exact = - self.alpha * u_exact
        y_d = 1 / (2 * math.pow(math.pi,2)) * temp
        return u_exact, y_exact, p_exact, y_d
    
    
    def loss(self,x):
        g = x
        g.requires_grad = True
        

        fi = self.forward(g)
        
        
        y = fi @ self.omega 
        p = fi @ self.omega 
        u = - 1/self.alpha * p
        
        x1, x2 = x[:,0][:,None],x[:,1][:,None]
        temp = torch.sin(math.pi* x1)* torch.sin(math.pi* x2)
        
        y_x = autograd.grad(y,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph=True,create_graph=True)[0]
        y_x1 = y_x[:,[0]]
        y_x2 = y_x[:,[1]]
        y_x1_x1x2 = autograd.grad(y_x1,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        y_x1x1 = y_x1_x1x2[:,[0]]
        y_x2_x1x2 = autograd.grad(y_x2,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        y_x2x2 = y_x2_x1x2[:,[1]]
        laplace_y = y_x1x1 + y_x2x2

        
                                
        p_x = autograd.grad(p,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph=True, create_graph=True)[0]
        p_x1 = p_x[:,[0]]
        p_x2 = p_x[:,[1]]
        p_x1_x1x2 = autograd.grad(p_x1,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        p_x1x1 = p_x1_x1x2[:,[0]]
        p_x2_x1x2 = autograd.grad(p_x2,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        p_x2x2 = p_x2_x1x2[:,[1]]
        
        laplace_p = p_x1x1 + p_x2x2
        
        y_d = 1 / (2 * math.pow(math.pi,2)) * temp

        
        r_p = - laplace_p   - y + y_d
    
        PDE = - laplace_y  - u

         
        loss_p = self.loss_function(r_p, torch.zeros(r_p.shape).to(self.device))
        loss_PDE = self.loss_function(PDE, torch.zeros(PDE.shape).to(self.device))
    
        return loss_p ,loss_PDE 
                                
                                
                                
    
                                
                                
    def train_model_adam(self, optimizer, x, n_epoch):
        if not torch.is_tensor(x):
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)

        iter_s = 0
        epsilon = 1e-3   
        gamma = 0.9
        eps = 1e-8
        w_min = 0.3
        w_max = 0.7
        beta = 0.5

        ema_p, ema_PDE = 1.0, 1.0
        initial_loss = None

        while iter_s < n_epoch:
            optimizer.zero_grad()
            loss_p, loss_PDE = self.loss(x)

            
            val_p = float(loss_p.detach())
            val_PDE = float(loss_PDE.detach())

            
            new_ema_p = gamma * ema_p + (1 - gamma) * val_p
            new_ema_PDE = gamma * ema_PDE + (1 - gamma) * val_PDE

            
            r1 = max((ema_p - new_ema_p) / (ema_p + eps), 0.0)
            r2 = max((ema_PDE - new_ema_PDE) / (ema_PDE + eps), 0.0)

            
            raw_w1 = 1.0 / ((r1 + eps) ** beta)
            raw_w2 = 1.0 / ((r2 + eps) ** beta)

           
            w1 = raw_w1 / (raw_w1 + raw_w2)

           
            w1 = w_min + (w_max - w_min) * w1
            w2 = 1.0 - w1

           
            loss = w1 * loss_p +  w2 * loss_PDE
            loss1 = loss_p + loss_PDE

            
            if iter_s == 0:
                initial_loss = float(loss1.detach())
                if initial_loss < 1e-12:   
                    print("Stop")
                    break

            
            if float(loss1.detach()) / initial_loss < epsilon:
                print("Stop:", iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()))
                break

            
            loss.backward()
            optimizer.step()

            
            ema_p, ema_PDE = new_ema_p, new_ema_PDE

            
            if iter_s == 0 or iter_s % 100 == 0:
                print(iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()))

            iter_s += 1

In [ ]:
# === Sampling ===
def sample_domain(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    xx, yy = np.meshgrid(x, y)
    xy = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    x_train = torch.tensor(xy, dtype=torch.float64)
    return x_train
def sample_boundary(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    points = []
    for xi in x:
        points.append([xi, 0])
        points.append([xi, 1])
    for yi in y:
        points.append([0, yi])
        points.append([1, yi])
    x_b_train = torch.tensor(points, dtype=torch.float64)
    return x_b_train

In [ ]:
M = 200
layers1 = np.array([2, 100, 100, 100, 100, M]) 

SNN = SNN(layers1, device)

SNN.to(device)
#print(PINN)
optimizer = torch.optim.Adam(SNN.parameters(), lr=0.001)
x_train = sample_domain(n=32)
start = timer()
SNN.train_model_adam(optimizer, x_train, 1e3)

In [ ]:
phi = SNN.forward(x_train)
Aphi_list = []
for j in range(M):
    phi_j = phi[:, j:j+1]
    grads = autograd.grad(phi_j, x_train, torch.ones_like(phi_j), create_graph=True)[0]
    d2x = autograd.grad(grads[:, 0], x_train, torch.ones_like(grads[:, 0]), create_graph=True)[0][:, 0:1]
    d2y = autograd.grad(grads[:, 1], x_train, torch.ones_like(grads[:, 1]), create_graph=True)[0][:, 1:2]
    Aphi_list.append(-(d2x + d2y))
A_phi = torch.cat(Aphi_list, dim=1)  # [N, M]

u_exact, y_exact, p_exact, y_d = SNN.exact_solution(x_train)
omega_p = torch.ones((M, 1), dtype=torch.float64).to(device)
omega_y = torch.ones((M, 1), dtype=torch.float64).to(device)
x_b_train = sample_boundary(n=32)
phi_x_b = SNN.forward(x_b_train)
u_b_exact, y_b_exact, p_b_exact, y_b_d = SNN.exact_solution(x_b_train)
rhs_y_b = y_b_exact
rhs_p_b = p_b_exact

A_aug = torch.cat([A_phi, phi_x_b], dim=0)  # [N + Nb, M]

In [ ]:
epoch = 20
i = 0
alpha = 1e-2
while i < epoch:
    rhs_y = - 1/alpha * phi @ omega_p
    b_y_aug = torch.cat([rhs_y, rhs_y_b], dim=0) 
    omega_y = torch.linalg.lstsq(A_aug, b_y_aug).solution
    #print(omega_y.shape)
    rhs_p = phi @ omega_y - y_d
    b_p_aug = torch.cat([rhs_p, rhs_p_b], dim=0) 
    omega_p = torch.linalg.lstsq(A_aug, b_p_aug).solution
    i += 1
    #print(i)
end = timer()
print("consumed time: " + str(end - start) + "s")

In [ ]:
def test(x_test):

        if torch.is_tensor(x_test) != True:
            x_test = torch.from_numpy(x_test).to(dtype=torch.float64).to(self.device)
            
        u,y,p,y_d = SNN.exact_solution(x_test)

        phi_pred = SNN.forward(x_test)
        y_pred = phi_pred @ omega_y
        p_pred = phi_pred @ omega_p
        u_pred = -1/SNN.alpha * p_pred
    
        error_y = torch.linalg.norm((y - y_pred),2)/torch.linalg.norm(y,2)
        y_pred = y_pred.cpu().detach().numpy()
        
        error_u = torch.linalg.norm((u - u_pred),2)/torch.linalg.norm(u,2)
        u_pred = u_pred.cpu().detach().numpy()
        
    
        return error_y, y_pred, y,error_u, u_pred, u

In [ ]:
x_test = sample_domain(n=100).to(device)
error_y, y_pred, y ,error_u, u_pred, u= test(x_test)
print('Test Error y: {:.2e}'.format(error_y))
print('Test Error u: {:.2e}'.format(error_u))